**_Windows Function_** -->  perform calculations across a group of rows while **keeping every row** in the output.

**_groupBy()_** --> aggregate sthe rows into fewer records, thus **reducing** the number of rows.

    custId    txnDate       amount
    1001      2026-01-01     100
    1001      2026-02-01     200
    1001      2026-03-01     300

    groupby("custId")

this collapse **MULTIPLE ROWS** into **ONE ROW PER CUSTOMER**. Here with groupBy() we lost teh transaction details as it fetched ONE row for 1001. 
with Windows - Spark calculates something **within each customer's group**
````
1001  2026-01-01  100   3
1001  2026-02-01  200   2
1001  2026-03-01  300   1
````




In [0]:
%sql
use catalog workspace;
use schema customer360;

select current_database(), current_catalog(), current_schema();

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

transactions = [
    (9001, 1001, "2026-01-10", 75.00),
    (9002, 1001, "2026-02-15", 120.00),
    (9003, 1001, "2026-03-20", 95.00),

    (9004, 1002, "2026-01-05", 200.00),
    (9005, 1002, "2026-03-01", 150.00),

    (9006, 1003, "2026-02-10", 80.00),
    (9007, 1003, "2026-03-15", 110.00)
]

trans_columns = ["txnId","custId","txnDate","amount"]

dfTransactions = spark.createDataFrame(transactions, trans_columns)
dfTransactions = dfTransactions.withColumn("txnDate", to_date(col("txnDate")))

display(dfTransactions)

In [0]:
CustomerWindow = (
    Window
        .partitionBy("custId")
        .orderBy(desc("txnDate"))
       )

dfRanked = dfTransactions.withColumn(
    "rowNum",
    row_number().over(CustomerWindow)
)
display(dfRanked)

**row_number()**
- assigns Unique sequential number within **each partition** accourding to specified ordering. 
- getlatest record per key
- Similar to SQL row_number()

**rank()**
if 2 rows has same rnk , it skips the next number (1,2,2,4)

**dense_rank()**
similar to rank() but does not skip the next number. (1,2,2,3)


In [0]:
CustomerWindow = (
    Window
        .partitionBy("custId")
        .orderBy(desc("txnDate"))
       )

dfRanked = dfTransactions.withColumn(
    "rowNum",
    row_number().over(CustomerWindow)
)

dfRanked = dfRanked.filter(col("rowNum") ==1)
display(dfRanked)

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

ranking_data = [
    (1001, 500),
    (1001, 400),
    (1001, 400),
    (1001, 300)
]

dfRanking = spark.createDataFrame(
    ranking_data,
    ["custId", "amount"]
)

display(dfRanking)

In [0]:
rnkpartition = (
    Window
    .partitionBy("custId")
    .orderBy(desc("amount"))
)

dfRanking = dfRanking.withColumn("row_number", row_number().over(rnkpartition)) \
    .withColumn("rank", rank().over(rnkpartition)) \
    .withColumn("dense_rank", dense_rank().over(rnkpartition))

display(dfRanking)

### Deduplicate Concept
First add rowNum and then do rank/row_number() to filter out invalid data

Exact duplicate rows
        ↓
    **distinct/dropDuplicates**

Multiple versions of same business entity
        ↓
    **Window + row_number**

In [0]:
dfRanking = dfRanking.withColumn("row_number", row_number().over(rnkpartition)) \
    .withColumn("rank", rank().over(rnkpartition)) \
    .withColumn("dense_rank", dense_rank().over(rnkpartition))

display(dfRanking)

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

customer_updates = [
    (1001, "John Smith", "Active", "2026-01-01 10:00:00", 1),
    (1001, "John Smith", "Active", "2026-02-15 09:30:00", 2),
    (1001, "John Smith", "Suspended", "2026-03-20 14:00:00", 3),

    (1002, "Mary Jones", "Active", "2026-01-10 08:00:00", 1),
    (1002, "Mary Jones", "Inactive", "2026-03-10 16:30:00", 3),

    (1003, "Steve Brown", "Active", "2026-02-01 12:00:00", 2)
]

dfUpdates = spark.createDataFrame(
    customer_updates,
    ["custId", "custName", "status", "updateTimestamp", "billCycle"]
)

dfUpdates = dfUpdates.withColumn(
    "updateTimestamp",
    to_timestamp("updateTimestamp")
)

display(dfUpdates)

### levels-
**partitionBy() --> orderBy() --> function() - row_number/dense_rank -->filter() --> display(filterData)**

In [0]:
custPartition = (Window
                 .partitionBy("custId")
                  .orderBy(desc("updateTimestamp"))
                )

dfUpdates = dfUpdates.withColumn(
    "rnk",
    row_number().over(custPartition)
)

dflatestRecord = dfUpdates.filter(col("rnk") ==1)

display(dflatestRecord)

### lag() --> to get previous status value


In [0]:
custPartition = (Window
                 .partitionBy("custId")
                  .orderBy(asc("updateTimestamp"))
                )

dfUpdates = dfUpdates.withColumn("prev_status",lag(col("status"), 1).over(custPartition))

dfFiltered = dfUpdates.filter(col("custId") == 1001)
display(dfFiltered)

#display(dfUpdates)

### lead() --> lead fetch next number

In [0]:
custPartition = (Window
                 .partitionBy("custId")
                  .orderBy(asc("updateTimestamp"))
                )

dfUpdates = dfUpdates.withColumn("NextBillCycle",lead(col("billCycle"), 1).over(custPartition))

dfFiltered = dfUpdates.filter(col("custId") == 1001)
display(dfFiltered)

#display(dfUpdates)

## Mini Challenge

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

customer_cdc = [
     (1001, "John Smith", "Active",    "2026-01-10 08:00:00"),
    (1001, "John Smith", "Active",    "2026-02-10 08:00:00"),
    (1001, "John Smith", "Suspended", "2026-03-15 10:00:00"),

    (1002, "Mary Jones", "Active",    "2026-01-01 09:00:00"),
    (1002, "Mary Jones", "Inactive",  "2026-04-01 11:00:00"),

    (1003, "Steve Brown", "Active",   "2026-02-20 12:00:00"),
    (1003, "Steve Brown", "Active",   "2026-03-20 12:00:00")
]

custCDC_columns = ["custId", "custName", "status", "updateTimestamp"]

dfCustomerCDC = spark.createDataFrame(customer_cdc, custCDC_columns)
dfCustomerCDC = dfCustomerCDC.withColumn("updateTimestamp",to_timestamp("updateTimestamp"))

display(dfCustomerCDC)

In [0]:
custPartition = (Window
                 .partitionBy("custId")
                  .orderBy(desc("updateTimestamp"))
                )

dfCustomerCDC = dfCustomerCDC.withColumn("rnk", row_number().over(custPartition))

dfCurrentCustomerStatus = dfCustomerCDC.filter(col("rnk") == 1)
dfCurrentCustomerStatus.write.mode("overwrite").saveAsTable("workspace.customer360.current_customer_status")

display(dfCurrentCustomerStatus)

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

customer_cdc = [
     (1001, "John Smith", "Active",    "2026-01-10 08:00:00"),
    (1001, "John Smith", "Active",    "2026-02-10 08:00:00"),
    (1001, "John Smith", "Suspended", "2026-03-15 10:00:00"),

    (1002, "Mary Jones", "Active",    "2026-01-01 09:00:00"),
    (1002, "Mary Jones", "Inactive",  "2026-04-01 11:00:00"),

    (1003, "Steve Brown", "Active",   "2026-02-20 12:00:00"),
    (1003, "Steve Brown", "Active",   "2026-03-20 12:00:00")
]

custCDC_columns = ["custId", "custName", "status", "updateTimestamp"]

dfCustomerCDC1 = spark.createDataFrame(customer_cdc, custCDC_columns)
dfCustomerCDC1 = dfCustomerCDC1.withColumn("updateTimestamp",to_timestamp("updateTimestamp"))

custPartition = (Window
                 .partitionBy("custId")
                  .orderBy(("updateTimestamp"))
                )

dfPrevStatus = dfCustomerCDC1.withColumn("prev_status", lag(col("status"),1).over(custPartition)) 

dfActualStatusChange = ( dfPrevStatus
                        .filter(col("prev_status").isNotNull()) 
                        .filter(col("status") != col("prev_status"))
                       )

display(dfActualStatusChange.select("custId", "custName", "status", "updateTimestamp", "prev_status"))